In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_timestamp

# 가짜 데이터로 실버/격리 테이블 시뮬레이션
silver_data = [
    (101, "2026-06-02 13:05:00", "UserA", 100, 120, False),
    (102, "2026-06-02 13:20:00", "UserB", 50, 60, False),
    (104, "2026-06-02 14:15:00", "UserD", 200, 210, True)
]
silver_schema = ["id", "timestamp", "user", "length_old", "length_new", "is_bot"]
mock_silver_df = spark.createDataFrame(silver_data, schema=silver_schema) \
                       .withColumn("timestamp", to_timestamp(col("timestamp")))

quarantine_data = [
    (103, "2026-06-02 13:45:00", "BadUser", 10, -5, "NEGATIVE_LENGTH"),
    (105, "2026-06-02 14:30:00", "SpamBot", 0, 500, "AI_SPAM_DETECTED")
]
quarantine_schema = ["id", "timestamp", "user", "length_old", "length_new", "failed_rule_name"]
mock_quarantine_df = spark.createDataFrame(quarantine_data, schema=quarantine_schema) \
                           .withColumn("timestamp", to_timestamp(col("timestamp")))

# 데이터가 잘 만들어졌나 확인해보기
print("--- 가짜 실버 레이어 데이터 ---")
display(mock_silver_df)

In [0]:
from pyspark.sql.functions import window, sum as _sum, count as _count, round as _round

# 1. 두 테이블 통합 (성공/실패 태그 부착)
passed_tagged = mock_silver_df.select("timestamp", lit(1).alias("is_passed"), lit(0).alias("is_dirty"))
failed_tagged = mock_quarantine_df.select("timestamp", lit(0).alias("is_passed"), lit(1).alias("is_dirty"))
combined_df = passed_tagged.union(failed_tagged)

# 2. 1시간 단위 실시간 윈도우 집계
gold_global_metrics_df = combined_df.groupBy(
    window(col("timestamp"), "1 hour").alias("time_window"),
    lit("WIKIPEDIA").alias("domain_name")
).agg(
    _count("is_passed").alias("total_ingested_rows"),
    _sum("is_passed").alias("passed_rows"),
    _sum("is_dirty").alias("quarantined_rows")
)

# 3. 데이터 순도 계산 및 최종 컬럼 정리
gold_final_df = gold_global_metrics_df.withColumn(
    "data_purity_rate",
    _round((col("passed_rows") / col("total_ingested_rows")) * 100, 2)
).select(
    col("time_window.start").alias("window_start"),
    col("domain_name"),
    col("total_ingested_rows"),
    col("passed_rows"),
    col("quarantined_rows"),
    col("data_purity_rate")
).orderBy("window_start")

# 최종 골드 레이어 관제탑 테이블 확인!
print("--- 최종 가공된 골드 레이어 통합 품질 매트릭스 ---")
display(gold_final_df)

In [0]:
# 1. 기업에게 보낼 최종 배송 경로 지정 (테스트용 임시 경로)
# 실무에서는 "dbfs:/mnt/export/현대자동차/clean_data" 같은 ADLS Gen2 경로가 됩니다.
delivery_path = "/tmp/export/wiki_clean_data"

# 2. 실버 레이어의 깨끗한 데이터를 CSV 파일로 추출하기
# .option("header", "true") -> 맨 위에 컬럼명을 이쁘게 적어달라는 뜻
# .mode("overwrite") -> 매일 새로 정제된 파일로 깔끔하게 덮어쓰겠다는 뜻
mock_silver_df.write \
              .format("csv") \
              .option("header", "true") \
              .mode("overwrite") \
              .save(delivery_path)

print(f"🎉 기업 고객용 청정 데이터 파일이 {delivery_path} 경로로 성공적으로 추출되었습니다!")

In [0]:
# 데이트브릭스 파일 시스템(dbutils)을 이용해 배송 폴더 내부 들여다보기
files = dbutils.fs.ls("/tmp/export/wiki_clean_data")

# 파일 목록 출력
for file in files:
    if file.name.endswith(".csv"):
        print(f"📄 내보내기 완료된 실제 파일명: {file.name}")

In [0]:
# /tmp 경로에 저장된 CSV 파일을 다시 읽어와서 화면에 출력
exported_file_df = spark.read \
                        .option("header", "true") \
                        .option("inferSchema", "true") \
                        .csv("/tmp/export/wiki_clean_data")

print("👀 실제 저장된 CSV 파일 내부 데이터 확인:")
display(exported_file_df)

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. 기업 고객이 격리실(Quarantine)에서 ID 103번 데이터를 찾아 올바르게 수정한 상황을 가정
# (길이 오타 -5를 정상 데이터 15로 수정)
fixed_data = [(103, "2026-06-02 13:45:00", "BadUser", 10, 15, False)] # 에러 유발 요소 해결!
columns = ["id", "timestamp", "user", "length_old", "length_new", "is_bot"]

fixed_df = spark.createDataFrame(fixed_data, schema=columns) \
                .withColumn("timestamp", to_timestamp(col("timestamp")))

# 2. [심폐소생] 수정된 데이터를 기존의 깨끗한 실버 테이블(mock_silver_df)에 합치기 (Union)
# 이제 103번 데이터는 더러운 데이터가 아니라 깨끗한 데이터가 되어 실버실로 이사 갑니다.
updated_silver_df = mock_silver_df.union(fixed_df)

# 3. 격리실(Quarantine) 테이블에서는 구출해냈으므로 103번을 제외(Delete 효과)시킵니다.
remaining_quarantine_df = mock_quarantine_df.filter(col("id") != 103)

# 4. 결과 확인
print("🎉 [구출 완료] 103번 데이터가 추가된 최종 실버 레이어:")
display(updated_silver_df.orderBy("id"))

print("🚧 현재 격리실에 남아있는 미해결 에러 데이터:")
display(remaining_quarantine_df)

In [0]:
# 1. 우리 프로젝트 전용 데이터베이스(스키마) 생성
# 명칭은 자유롭게 변경 가능합니다. 여기서는 'data_quality_pjt'로 지정합니다.
spark.sql("CREATE DATABASE IF NOT EXISTS data_quality_pjt")
spark.sql("USE data_quality_pjt")

# 2. 실버 레이어 영구 테이블로 저장 (.saveAsTable)
# 이 명령어를 쓰면 ADLS Gen2 스토리지에 실제 파일이 수스루 쌓이고 관리형 테이블로 등록됩니다.
mock_silver_df.write \
              .format("delta") \
              .mode("overwrite") \
              .saveAsTable("data_quality_pjt.silver_wiki_structured")

# 3. 격리 레이어 영구 테이블로 저장
mock_quarantine_df.write \
                  .format("delta") \
                  .mode("overwrite") \
                  .saveAsTable("data_quality_pjt.quarantine_wiki_structured")

print("✨ [물리 창고 구축 완료] silver_wiki_structured와 quarantine_wiki_structured 테이블이 데이터베이스에 영구 등록되었습니다!")

In [0]:
from pyspark.sql.functions import current_date, desc
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# 1. 방금 물리적으로 저장한 격리실 Delta 테이블을 다이렉트로 읽어옵니다.
quarantine_table_df = spark.read.table("data_quality_pjt.quarantine_wiki_structured")

# 2. 날짜별, 도메인별, 에러 규칙별로 위반 횟수(violation_count) 집계
error_rank_df = quarantine_table_df.groupBy(
    current_date().alias("base_date"),
    F.lit("WIKIPEDIA").alias("domain_name"),
    F.col("failed_rule_name")
).agg(
    F.count("id").alias("violation_count")
)

# 3. 윈도우 함수를 써서 에러 빈도 순위(Rank) 매기기
windowSpec = Window.partitionBy("domain_name").orderBy(desc("violation_count"))
final_error_rank_df = error_rank_df.withColumn("error_rank", F.dense_rank().over(windowSpec))

# 4. 최종 2호 골드 테이블을 실제 Delta 테이블로 영구 저장
final_error_rank_df.write \
                   .format("delta") \
                   .mode("overwrite") \
                   .saveAsTable("data_quality_pjt.agg_domain_error_top_rules")

print("🏆 [Gold 2호 테이블 완공] 일자별 에러 순위 집계 테이블이 생성되었습니다.")
display(spark.read.table("data_quality_pjt.agg_domain_error_top_rules"))

In [0]:
%sql
-- 아키텍트가 구축한 데이터베이스 내부의 테이블 목록 확인
SHOW TABLES IN data_quality_pjt;

In [0]:
%sql
-- 2호 골드 테이블에 순위가 잘 매겨졌는지 조회
SELECT * FROM data_quality_pjt.agg_domain_error_top_rules;

In [0]:
# 어제 만들어둔 골드 데이터프레임(gold_final_df)을 
# data_quality_pjt 데이터베이스에 'agg_global_data_quality_metrics'라는 이름으로 영구 저장합니다.
gold_final_df.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable("data_quality_pjt.agg_global_data_quality_metrics")

print("🏆 [Gold 1호 테이블 완공] 이제 물리 테이블이 존재합니다. 뷰를 만들러 가셔도 됩니다!")

In [0]:
%sql
-- 1. 기존에 data_quality_pjt 데이터베이스를 사용하도록 설정
USE data_quality_pjt;

-- 2. 웹 대시보드 메인 차트용 가상 뷰(View) 생성
CREATE OR REPLACE VIEW data_quality_pjt.vw_web_main_dashboard AS
SELECT 
    window_start,                    -- 1시간 단위 시작 시간
    domain_name,                     -- 도메인 (예: WIKIPEDIA)
    total_ingested_rows AS total_cnt, -- 총 유입량 (웹 개발자가 보기 편하게 컬럼명 변경)
    passed_rows AS clean_cnt,        -- 통과량
    quarantined_rows AS error_cnt,   -- 격리량
    data_purity_rate AS purity_rate  -- 데이터 순도(%)
FROM 
    data_quality_pjt.agg_global_data_quality_metrics;

In [0]:
%sql
-- 3. 에러 원인 분석 차트용 가상 뷰(View) 생성
CREATE OR REPLACE VIEW data_quality_pjt.vw_web_error_ranking AS
SELECT 
    base_date,          -- 기준 일자
    domain_name,        -- 도메인
    failed_rule_name,   -- 위반한 규칙명
    violation_count,    -- 위반 건수
    error_rank          -- 에러 순위 (1위, 2위...)
FROM 
    data_quality_pjt.agg_domain_error_top_rules
WHERE 
    error_rank <= 5;    -- 웹 화면 복잡도를 낮추기 위해 TOP 5만 쏙 골라내기

In [0]:
%sql
-- 웹 백엔드 개발자가 우리 데이터베이스에 접속해서 날릴 쿼리를 미리 시뮬레이션 합니다.
SELECT * FROM data_quality_pjt.vw_web_main_dashboard;